**Author:** Adrian Vanyi 

# Strategy Comparison

This notebook implements the comparison methodology of §14 of the documentation. It sweeps the four strategies (momentum, factors model, MVO max-return, and MVO min-variance) across a grid of 12 start dates and strategy return targets, with all other parameters held fixed (see §14.2). It then isolates the momentum strategy to measure the effect of enabling the dynamic rebalancing rule.

Two tables are produced:
- **Table A**: strategy × return-target sweep across the four strategies, with the dynamic rebalancing rule disabled.
- **Table B**: return-target sweep on the momentum strategy alone, comparing runs with the dynamic rebalancing rule enabled against runs with it disabled.

The notebook also runs the backtests to compute the Sharpe ratios (daily and annualized), with and without the dynamic rebalancing rule enabled.

**Prerequisite:** The data files (S&P 500 historical members, prices, dividends, SPY daily returns, EFFR) must already be present in `../data/`. If they are not, run `python download_data.py` from the project root first.

### Notebook structure
0. Setup
1. Load pre-downloaded data (and filter out bad tickers)
2. Compute risk-free returns
3. Configure the comparison grid
4. Build Table A
5. Termination causes by strategy (across all Table A runs)
6. Build Table B
7. Termination causes with the dynamic rebalancing rule enabled on the momentum strategy (across all Table B runs)
8. Query a specific result across all backtest runs (Table A or B)
9. Compute Sharpe ratios (daily, annualized) of each strategy for each historical window

## 0. Setup

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import logging
import numpy as np
import pandas as pd
from tqdm.std import tqdm

from modules import logging_setup
from modules import rates as r
from modules import universe_construction as uc
from modules import strategies_comparison as sc
from modules import kpis as kpis


# Comparison runs many backtests, so we keep logging at ERROR to avoid clutter.
logging_setup.configure(level="ERROR",  stream = sys.stdout)
logger = logging.getLogger("strategies_comparison")
logger.setLevel(logging.ERROR)


DATA_DIR = PROJECT_ROOT / "data"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
OUTPUTS_DIR.mkdir(exist_ok=True)

## 1. Load pre-downloaded data (and filter out bad tickers)

*(The filtering out of bad tickers is explained in §6.5 and the Appendix of the documentation*.)

In [3]:
expected_files = [
    "sp500_members_at_start_of_months.parquet",
    "prices_sp500_members.parquet",
    "dividends_data.parquet",
    "daily_spy_returns.parquet",
    "EFFR.csv",
]

missing = [f for f in expected_files if not (DATA_DIR / f).exists()]

if missing:
    raise FileNotFoundError(
        f"data files missing in {DATA_DIR}: {missing}. "
        f"Run `python download_data.py` from the project root first."
    )

sp500_members_df = pd.read_parquet(DATA_DIR / "sp500_members_at_start_of_months.parquet")

first_trading_day_of_month_to_sp500_members_dict = {
    idx: row.dropna().tolist() for idx, row in sp500_members_df.iterrows()
}

price_data_raw = pd.read_parquet(DATA_DIR / "prices_sp500_members.parquet")

dividend_data = pd.read_parquet(DATA_DIR / "dividends_data.parquet").squeeze()

daily_spy_returns = pd.read_parquet(DATA_DIR / "daily_spy_returns.parquet").squeeze()

effr = (
    pd.read_csv(
        DATA_DIR / "EFFR.csv",
        parse_dates=["observation_date"],
        date_format="%d/%m/%Y",
        index_col=0,
    ).squeeze()
    / 100
)
effr = effr.ffill()

# --- Filter-out bad tickers
price_data, bad_tickers = uc.remove_bad_tickers(price_data_raw)

all_tickers_during_backtest = (
    price_data.index.get_level_values("ticker").unique().to_list()
)

logger.info(
    "loaded data: %s tickers (dropped %s bad), %s price rows",
    len(all_tickers_during_backtest), len(bad_tickers), len(price_data),
)

## 2. Compute risk-free returns

In [4]:
r_d = effr / r.ACT_360

calendar_days = (
    pd.Series(r_d.index, index=r_d.index)
    - pd.Series(r_d.index, index=r_d.index).shift(1)
).dt.days

rf_daily_returns = (
    (r_d.shift(1) * calendar_days)
    .rename("rf_return_since_previous_trading_date")
)

rf_daily_returns.index.name = "date"

## 3. Configure the comparison grid

*(Using the parameters specified in §14.2.2 of the documentation)*

In [5]:
config = sc.FixedBacktestConfig(
                # Rebalance cadence and horizon
                rebalance_freq_type = "ndays",
                rebalance_frequency_trading_days = 23,
                number_of_inter_rebalance_periods = 12, 
            
                # Estimation windows
                rolling_periods_for_cov_matrix_estimation = 18,
                rolling_periods_for_beta_regression = 12,
                rolling_periods_for_avg_return_computation = 12,
                rolling_periods_for_factors_model_regression = 6,
                rolling_window_trading_days_for_volatility = 60,
                rolling_window_trading_days_for_momentum = 4 * 23,
                buffer_trading_days_for_momentum = 23,
            
                initial_cash = 100_000.0,
                pct_initial_equity_triggering_stop_loss = 0.5,
                cure_method_for_mr_violation = "shrinking_exposures",
            
                # Inter-rebalance return target used when the rule is enabled
                inter_rebalance_target = 0.03/12 
            )

# Pre-loaded data shared across all runs.
data_bundle = sc.BacktestDataBundle(
                    price_data = price_data,
                    dividend_data = dividend_data,
                    daily_spy_returns = daily_spy_returns,
                    rf_daily_returns = rf_daily_returns,
                    effr = effr,
                    first_trading_day_of_month_to_sp500_members_dict=(
                        first_trading_day_of_month_to_sp500_members_dict
                    ),
                    all_tickers_during_backtest = all_tickers_during_backtest,
                )

START_DATES = [
    pd.Timestamp("2021-01-04"),
    pd.Timestamp("2021-04-05"),
    pd.Timestamp("2021-07-06"),
    pd.Timestamp("2021-10-04"),
    pd.Timestamp("2022-01-03"),
    pd.Timestamp("2022-04-04"),
    pd.Timestamp("2022-07-05"),
    pd.Timestamp("2022-10-03"),
    pd.Timestamp("2023-01-03"),
    pd.Timestamp("2023-04-03"),
    pd.Timestamp("2023-07-03"),
    pd.Timestamp("2023-10-02"),
]

RETURN_TARGETS = [0.01,
                  0.03,
                  0.05,
                  0.07, 
                  0.10,
                  0.12
                 ]

STRATEGY_NAMES_TABLE_A = [
    "momentum",
    "factors_model",
    "mvo_max_return",
    "mvo_min_variance"
]

logger.info(
    "comparison grid: %d start dates × %d return targets × %d strategies = %d Table-A runs; "
    "+ %d × %d × 2 = %d Table-B runs",
    len(START_DATES), len(RETURN_TARGETS), len(STRATEGY_NAMES_TABLE_A),
    len(START_DATES) * len(RETURN_TARGETS) * len(STRATEGY_NAMES_TABLE_A),
    len(START_DATES), len(RETURN_TARGETS),
    len(START_DATES) * len(RETURN_TARGETS) * 2,
)

## 4. Build Table A 


In [6]:
table_a_records: list[sc.RunResult] = []
table_a_total = len(START_DATES) * len(RETURN_TARGETS) * len(STRATEGY_NAMES_TABLE_A)

with tqdm(total=table_a_total, desc="Backtest runs completed (for Table A)", file=sys.stdout) as pbar:
    for start in START_DATES:
        for target in RETURN_TARGETS:
            for strategy_name in STRATEGY_NAMES_TABLE_A:
                result = sc.run_one_backtest(
                            start_date = start,
                            strategy_name = strategy_name,
                            use_return_target_hit_rule = True,
                            return_target = target,
                            use_inter_rebalance_rule = False,
                            inter_rebalance_target = None,
                            config = config,
                            data = data_bundle
                )
                table_a_records.append(result)
                pbar.update(1)
                pbar.refresh()

Backtest runs completed (for Table A): 100%|█████████████████████████████████████████| 288/288 [44:11<00:00,  9.21s/it]


**Raw table A**

In [7]:
table_a_raw = pd.DataFrame([res.__dict__ for res in table_a_records])

table_a_raw["time_to_target"] = table_a_raw.apply(
    lambda res: res["trading_days_run"] if res["target_hit"] else np.nan, axis=1,
)

n_errors = (table_a_raw["cause_of_termination"] == "error").sum()

print(f"Table A: {len(table_a_raw)} backtest runs completed ({n_errors} errors)")

table_a_raw.head()

Table A: 288 backtest runs completed (0 errors)


,start_date,strategy_name,return_target,use_inter_rebalance_rule,inter_rebalance_target,cause_of_termination,trading_days_run,total_return,target_hit,error,time_to_target
0,2021-01-04,momentum,0.01,False,None,hit_return_target_for_strategy,3,0.023968,True,None,3.0
1,2021-01-04,factors_model,0.01,False,None,hit_return_target_for_strategy,2,0.021277,True,None,2.0
2,2021-01-04,mvo_max_return,0.01,False,None,hit_return_target_for_strategy,2,0.015910,True,None,2.0
3,2021-01-04,mvo_min_variance,0.01,False,None,hit_return_target_for_strategy,4,0.010581,True,None,4.0
4,2021-01-04,momentum,0.03,False,None,stop_loss_termination,234,-0.542023,False,None,NaN


**Formatted Table A**

In [8]:
table_a = sc.aggregate_to_table(
    table_a_raw,
    group_col = "strategy_name",
    group_order = STRATEGY_NAMES_TABLE_A,
    return_targets = RETURN_TARGETS
)

**Display styled formatted table A**

In [9]:
sc.style_comparison_table(table_a)

**Save to disk**

In [10]:
table_a.to_parquet(OUTPUTS_DIR / "table_a.parquet")

## 5. Termination causes by strategy (across all Table A runs)

In [11]:
display(
    pd.crosstab(
        table_a_raw["strategy_name"],
        table_a_raw["cause_of_termination"],
        margins=True
    )
)

cause_of_termination,hit_return_target_for_strategy,reached_last_scheduled_backtest_date,stop_loss_termination,All
strategy_name,,,,
factors_model,43,8,21,72
momentum,61,1,10,72
mvo_max_return,57,0,15,72
mvo_min_variance,54,18,0,72
All,215,27,46,288


## 6. Build Table B 


In [12]:
table_b_records: list[sc.RunResult] = []
table_b_total = len(START_DATES) * len(RETURN_TARGETS) * 2

with tqdm(total=table_b_total, desc="Backtest runs completed (for table B)", file=sys.stdout) as pbar:
    for start in START_DATES:
        for target in RETURN_TARGETS:
            for use_rule in (True, False):
                result = sc.run_one_backtest(
                    start_date = start,
                    strategy_name = "momentum",
                    use_return_target_hit_rule = True,
                    return_target = target,
                    use_inter_rebalance_rule = use_rule,
                    inter_rebalance_target = config.inter_rebalance_target,
                    config = config,
                    data = data_bundle
                )
                table_b_records.append(result)
                pbar.update(1)

Backtest runs completed (for table B): 100%|█████████████████████████████████████████| 144/144 [25:33<00:00, 10.65s/it]


**Raw table B**

In [13]:
table_b_raw = pd.DataFrame([res.__dict__ for res in table_b_records])
table_b_raw["time_to_target"] = table_b_raw.apply(
    lambda res: res["trading_days_run"] if res["target_hit"] else np.nan, axis=1,
)

n_errors = (table_b_raw["cause_of_termination"] == "error").sum()
print(f"Table B: {len(table_b_raw)} backtest runs completed ({n_errors} errors)")

table_b_raw.head()

Table B: 144 backtest runs completed (0 errors)


,start_date,strategy_name,return_target,use_inter_rebalance_rule,inter_rebalance_target,cause_of_termination,trading_days_run,total_return,target_hit,error,time_to_target
0,2021-01-04,momentum,0.01,True,0.0025,hit_return_target_for_strategy,3,0.023968,True,None,3.0
1,2021-01-04,momentum,0.01,False,NaN,hit_return_target_for_strategy,3,0.023968,True,None,3.0
2,2021-01-04,momentum,0.03,True,0.0025,hit_return_target_for_strategy,4,0.031406,True,None,4.0
3,2021-01-04,momentum,0.03,False,NaN,stop_loss_termination,234,-0.542023,False,None,NaN
4,2021-01-04,momentum,0.05,True,0.0025,hit_return_target_for_strategy,23,0.067457,True,None,23.0


**Formatted Table B: rule enabled/disabled  × return-target sweep**

In [14]:
table_b_raw_groups = table_b_raw.copy()
table_b_raw_groups["rule_setting"] = table_b_raw_groups["use_inter_rebalance_rule"].map(
    {True: "with_inter_rebalance_rule", False: "without_inter_rebalance_rule"}
)
table_b = sc.aggregate_to_table(
    table_b_raw_groups,
    group_col = "rule_setting",
    group_order = ["with_inter_rebalance_rule", "without_inter_rebalance_rule"],
    return_targets = RETURN_TARGETS
)

**Display styled formatted Table B**

In [15]:
sc.style_comparison_table(table_b)

**Save to disk**

In [16]:
table_b.to_parquet(OUTPUTS_DIR / "table_b.parquet")

## 7. Termination causes with the dynamic rebalancing rule enabled on the momentum strategy (across all Table B runs)

In [17]:
display(
    pd.crosstab(
        table_b_raw["use_inter_rebalance_rule"].map(
            {True: "with_rule", False: "without_rule"}
        ),
        table_b_raw["cause_of_termination"],
        margins=True,
    )
)

cause_of_termination,hit_return_target_for_strategy,reached_last_scheduled_backtest_date,stop_loss_termination,All
use_inter_rebalance_rule,,,,
with_rule,59,0,13,72
without_rule,61,1,10,72
All,120,1,23,144


## 8. Query a specific result across all backtest runs (Table A or B)

**In Table A**

In [18]:
mask = (
    (table_a_raw["strategy_name"] == "momentum")
    & (table_a_raw["cause_of_termination"] == "hit_return_target_for_strategy")
    & (table_a_raw["time_to_target"] <=10 )
    & (table_a_raw["return_target"] == 0.05)
)

table_a_raw[mask]

,start_date,strategy_name,return_target,use_inter_rebalance_rule,inter_rebalance_target,cause_of_termination,trading_days_run,total_return,target_hit,error,time_to_target
56,2021-07-06,momentum,0.05,False,None,hit_return_target_for_strategy,7,0.055010,True,None,7.0
128,2022-04-04,momentum,0.05,False,None,hit_return_target_for_strategy,3,0.068132,True,None,3.0
248,2023-07-03,momentum,0.05,False,None,hit_return_target_for_strategy,10,0.067436,True,None,10.0
272,2023-10-02,momentum,0.05,False,None,hit_return_target_for_strategy,8,0.077415,True,None,8.0


**In Table B**

In [19]:
mask = (
     (table_b_raw["cause_of_termination"] == "hit_return_target_for_strategy") 
    & (table_b_raw["time_to_target"] < 10)
    & (table_b_raw["use_inter_rebalance_rule"] == True)
    & (table_b_raw["return_target"] == 0.05)
)

table_b_raw[mask]

,start_date,strategy_name,return_target,use_inter_rebalance_rule,inter_rebalance_target,cause_of_termination,trading_days_run,total_return,target_hit,error,time_to_target
28,2021-07-06,momentum,0.05,True,0.0025,hit_return_target_for_strategy,6,0.052215,True,None,6.0
64,2022-04-04,momentum,0.05,True,0.0025,hit_return_target_for_strategy,3,0.088583,True,None,3.0
136,2023-10-02,momentum,0.05,True,0.0025,hit_return_target_for_strategy,6,0.053632,True,None,6.0


# 9. Compute Sharpe ratios (daily, annualized) of each strategy for each historical window

We set `rebalance_freq_type` to `"monthly"`, and we can build any specific historical window from a tuple (`start_date`, `number_of_inter_rebalance_periods`), as demonstrated in the notebook `backteste_calendar.ipynb`.

## With the dynamic rebalancing rule disabled

In [20]:
initial_cash= 100_000.0

HISTORICAL_WINDOWS = [(pd.Timestamp("2010-03-05"), 12*15-7),        # corresponds to 2010-03-05 to 2026-02-02 (full sample of daily returns)
                      (pd.Timestamp("2010-03-05"),  12*9-1),    # corresponds to 2010-03-05 to 2020-01-16 (post-GFC bull)
                      (pd.Timestamp("2022-10-05"), 12*3 -1)       # corresponds to 2022-10-05 to 2026-01-26 (post-Covid bull)
                     ]

STRATEGY_NAMES = [
    "momentum",
    "factors_model",
    "mvo_max_return",
    "mvo_min_variance"
]

INTER_REBALANCE_RULE = [False]

inter_rebalance_target = 0.03/12

sharpe_records_raw_with_dynamic_reb_disabled : list[sc.Results_Sharpe] = []

total_backtest_runs = len(HISTORICAL_WINDOWS) * len(STRATEGY_NAMES) * len(INTER_REBALANCE_RULE)


with tqdm(total=total_backtest_runs, desc="Backtest runs completed (for Sharpe computations)", file=sys.stdout) as pbar:
    for historical_window in HISTORICAL_WINDOWS:
        for strategy_name in STRATEGY_NAMES:
            for use_inter_rebalance_rule in INTER_REBALANCE_RULE:

                start_date, number_of_inter_rebalance_periods = historical_window
        
                config = sc.FixedBacktestConfig(
                                # Rebalance cadence and horizon
                                rebalance_freq_type = "ndays",
                                rebalance_frequency_trading_days = 23, 
                                number_of_inter_rebalance_periods = number_of_inter_rebalance_periods, 
                            
                                # Estimation windows
                                rolling_periods_for_cov_matrix_estimation = 18,
                                rolling_periods_for_beta_regression = 12,
                                rolling_periods_for_avg_return_computation = 12,
                                rolling_periods_for_factors_model_regression = 6,
                                rolling_window_trading_days_for_volatility = 60,
                                rolling_window_trading_days_for_momentum = 4 * 23,
                                buffer_trading_days_for_momentum = 23,
                            
                                initial_cash = initial_cash,
                                pct_initial_equity_triggering_stop_loss = 0.0,
                                cure_method_for_mr_violation = "shrinking_exposures",
                            
                                # Inter-rebalance return target used when the rule is enabled
                                inter_rebalance_target = inter_rebalance_target
                            )
            
                
                result = sc.compute_sharpe_ratio(
                            start_date = start_date,
                            strategy_name = strategy_name,
                            use_inter_rebalance_rule = use_inter_rebalance_rule,
                            config = config,
                            data = data_bundle
                )
                sharpe_records_raw_with_dynamic_reb_disabled.append(result)
                pbar.update(1)
                pbar.refresh()

Backtest runs completed (for Sharpe computations): 100%|██████████████████████████████| 12/12 [22:20<00:00, 111.71s/it]


**Build and format sharpe_records DataFrame**

In [21]:
sharpe_records_with_dynamic_reb_disabled = pd.DataFrame([res.__dict__ for res in sharpe_records_raw_with_dynamic_reb_disabled])

sharpe_table_with_dynamic_reb_disabled = sc.format_sharpe_records(sharpe_records_with_dynamic_reb_disabled)

**Display styled table**

In [22]:
sc.style_sharpe_table(sharpe_table_with_dynamic_reb_disabled)

**Save to disk**

In [23]:
sharpe_table_with_dynamic_reb_disabled.to_parquet(OUTPUTS_DIR / "sharpe_table_with_dynamic_reb_disabled.parquet")

## With the dynamic rebalancing rule enabled

In [ ]:
initial_cash= 100_000.0

HISTORICAL_WINDOWS = [(pd.Timestamp("2010-03-05"), 12*15-7),    # corresponds to 2010-03-05 to 2026-02-02 (full sample of daily returns)
                      (pd.Timestamp("2010-03-05"),  12*9-1),    # corresponds to 2010-03-05 to 2020-01-16 (post-GFC bull)
                      (pd.Timestamp("2022-10-05"), 12*3 -1)     # corresponds to 2022-10-05 to 2026-01-26 (bull)
                     ]

STRATEGY_NAMES = [
    "momentum",
    "factors_model",
    "mvo_max_return",
    "mvo_min_variance"
]

INTER_REBALANCE_RULE = [True]

inter_rebalance_target = 0.03/12

sharpe_records_raw_with_dynamic_reb_enabled : list[sc.Results_Sharpe] = []

total_backtest_runs = len(HISTORICAL_WINDOWS) * len(STRATEGY_NAMES) * len(INTER_REBALANCE_RULE)


with tqdm(total=total_backtest_runs, desc="Backtest runs completed (for Sharpe computations)", file=sys.stdout) as pbar:
    for historical_window in HISTORICAL_WINDOWS:
        for strategy_name in STRATEGY_NAMES:
            for use_inter_rebalance_rule in INTER_REBALANCE_RULE:

                start_date, number_of_inter_rebalance_periods = historical_window
        
                config = sc.FixedBacktestConfig(
                                # Rebalance cadence and horizon
                                rebalance_freq_type = "ndays",
                                rebalance_frequency_trading_days = 23, 
                                number_of_inter_rebalance_periods = number_of_inter_rebalance_periods, 
                            
                                # Estimation windows
                                rolling_periods_for_cov_matrix_estimation = 18,
                                rolling_periods_for_beta_regression = 12,
                                rolling_periods_for_avg_return_computation = 12,
                                rolling_periods_for_factors_model_regression = 6,
                                rolling_window_trading_days_for_volatility = 60,
                                rolling_window_trading_days_for_momentum = 4 * 23,
                                buffer_trading_days_for_momentum = 23,
                            
                                initial_cash = initial_cash,
                                pct_initial_equity_triggering_stop_loss = 0.0,
                                cure_method_for_mr_violation = "shrinking_exposures",
                            
                                # Inter-rebalance return target used when the rule is enabled
                                inter_rebalance_target = inter_rebalance_target
                            )
            
                
                result = sc.compute_sharpe_ratio(
                            start_date = start_date,
                            strategy_name = strategy_name,
                            use_inter_rebalance_rule = use_inter_rebalance_rule,
                            config = config,
                            data = data_bundle
                )
                sharpe_records_raw_with_dynamic_reb_enabled.append(result)
                pbar.update(1)
                pbar.refresh()

Backtest runs completed (for Sharpe computations): 100%|███████████████████████████| 12/12 [3:35:16<00:00, 1076.35s/it]


**Build and format sharpe_records DataFrame**

In [25]:
sharpe_records_with_dynamic_reb_enabled = pd.DataFrame([res.__dict__ for res in sharpe_records_raw_with_dynamic_reb_enabled])

sharpe_table_with_dynamic_reb_enabled = sc.format_sharpe_records(sharpe_records_with_dynamic_reb_enabled)

**Display styled table**

In [26]:
sc.style_sharpe_table(sharpe_table_with_dynamic_reb_enabled)

**Save to disk**

In [27]:
sharpe_table_with_dynamic_reb_enabled.to_parquet(OUTPUTS_DIR /  "sharpe_table_with_dynamic_reb_enabled.parquet")